<a href="https://colab.research.google.com/github/Ananya-mandal56/DWBDA_Assignment/blob/main/Assignment_2_MapReduce_Student_Grades_REVISED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 — MapReduce: Number of Students in Each Grade Category

**Objective:** Apply the MapReduce framework to calculate the number of students in each category **S, A, B, C, D, E and F** for all subjects.

## Dataset
An official previous-semester result dataset was not provided by the instructor. Therefore, this assignment uses a **synthetic/anonymized previous-semester-style final exam dataset** containing 30 students and 5 subjects (150 exam records).



## 1. Load the dataset

In [1]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

file_name = "previous_semester_final_exam_results.csv"
df = pd.read_csv("previous_semester_final_exam_results.csv")

print("Dataset shape:", df.shape)
display(df.head(10))

Saving previous_semester_final_exam_results.csv to previous_semester_final_exam_results.csv
Dataset shape: (150, 3)


,Student_ID,Subject,Grade
0,S001,Data Structures,B
1,S001,Database Management,E
2,S001,Computer Networks,A
3,S001,Operating Systems,D
4,S001,Python Programming,S
5,S002,Data Structures,D
6,S002,Database Management,S
7,S002,Computer Networks,C
8,S002,Operating Systems,F
9,S002,Python Programming,B


## 2. Check and clean the data

In [2]:
valid_grades = ["S", "A", "B", "C", "D", "E", "F"]

df["Student_ID"] = df["Student_ID"].astype(str).str.strip()
df["Subject"] = df["Subject"].astype(str).str.strip()
df["Grade"] = df["Grade"].astype(str).str.strip().str.upper()

# Keep only valid grade categories
df = df[df["Grade"].isin(valid_grades)].copy()

print("Students:", df["Student_ID"].nunique())
print("Subjects:", df["Subject"].nunique())
print("Exam records:", len(df))
print("Grade categories:", sorted(df["Grade"].unique()))
display(df.head())

Students: 30
Subjects: 5
Exam records: 150
Grade categories: ['A', 'B', 'C', 'D', 'E', 'F', 'S']


,Student_ID,Subject,Grade
0,S001,Data Structures,B
1,S001,Database Management,E
2,S001,Computer Networks,A
3,S001,Operating Systems,D
4,S001,Python Programming,S


## 3. Map phase

The Mapper converts each exam record into:

**(Subject, Grade) → 1**

For example:

`(Data Structures, A) → 1`

In [3]:
mapped = [
    ((row.Subject, row.Grade), 1)
    for row in df.itertuples(index=False)
]

print("First 15 mapper outputs:")
for item in mapped[:15]:
    print(item)

First 15 mapper outputs:
(('Data Structures', 'B'), 1)
(('Database Management', 'E'), 1)
(('Computer Networks', 'A'), 1)
(('Operating Systems', 'D'), 1)
(('Python Programming', 'S'), 1)
(('Data Structures', 'D'), 1)
(('Database Management', 'S'), 1)
(('Computer Networks', 'C'), 1)
(('Operating Systems', 'F'), 1)
(('Python Programming', 'B'), 1)
(('Data Structures', 'F'), 1)
(('Database Management', 'B'), 1)
(('Computer Networks', 'E'), 1)
(('Operating Systems', 'A'), 1)
(('Python Programming', 'D'), 1)


## 4. Reduce phase

The Reducer groups identical `(Subject, Grade)` keys and adds their values.

**(Subject, Grade) → total number of students**

In [4]:
reduced = {}

for key, value in mapped:
    reduced[key] = reduced.get(key, 0) + value

print("Reducer output:")
for key in sorted(reduced):
    print(key, "->", reduced[key])

Reducer output:
('Computer Networks', 'A') -> 4
('Computer Networks', 'B') -> 5
('Computer Networks', 'C') -> 4
('Computer Networks', 'D') -> 4
('Computer Networks', 'E') -> 5
('Computer Networks', 'F') -> 3
('Computer Networks', 'S') -> 5
('Data Structures', 'A') -> 5
('Data Structures', 'B') -> 4
('Data Structures', 'C') -> 5
('Data Structures', 'D') -> 4
('Data Structures', 'E') -> 4
('Data Structures', 'F') -> 5
('Data Structures', 'S') -> 3
('Database Management', 'A') -> 4
('Database Management', 'B') -> 5
('Database Management', 'C') -> 3
('Database Management', 'D') -> 5
('Database Management', 'E') -> 4
('Database Management', 'F') -> 5
('Database Management', 'S') -> 4
('Operating Systems', 'A') -> 5
('Operating Systems', 'B') -> 3
('Operating Systems', 'C') -> 5
('Operating Systems', 'D') -> 4
('Operating Systems', 'E') -> 5
('Operating Systems', 'F') -> 4
('Operating Systems', 'S') -> 4
('Python Programming', 'A') -> 5
('Python Programming', 'B') -> 4
('Python Programming',

## 5. Final result: number of students in each category for every subject

In [ ]:
result = pd.DataFrame(
    [
        {"Subject": subject, "Grade": grade, "Student_Count": count}
        for (subject, grade), count in reduced.items()
    ]
)

pivot_result = (
    result
    .pivot(index="Subject", columns="Grade", values="Student_Count")
    .fillna(0)
    .astype(int)
    .reindex(columns=valid_grades, fill_value=0)
)

display(pivot_result)

## 6. Overall grade count across all subjects

In [5]:
overall = (
    df["Grade"]
    .value_counts()
    .reindex(valid_grades, fill_value=0)
    .rename_axis("Grade")
    .reset_index(name="Student_Count")
)

display(overall)

,Grade,Student_Count
0,S,20
1,A,23
2,B,21
3,C,21
4,D,22
5,E,21
6,F,22


## 7. Verification

Each subject should contain 30 students because the synthetic dataset has 30 students taking every subject.

In [6]:
subject_totals = df.groupby("Subject")["Student_ID"].nunique()

print("Students per subject:")
display(subject_totals.to_frame("Number_of_Students"))

assert (subject_totals == 30).all()
print("Verification passed: every subject has 30 students.")

Students per subject:


,Number_of_Students
Subject,
Computer Networks,30
Data Structures,30
Database Management,30
Operating Systems,30
Python Programming,30


Verification passed: every subject has 30 students.


## Conclusion

The MapReduce framework was successfully implemented to count students in grade categories S, A, B, C, D, E and F for every subject. The Mapper emitted `(Subject, Grade) → 1`, and the Reducer summed the values for each key. The final table provides the grade distribution subject-wise.

**Dataset note:** Since no official previous-semester result dataset was provided, a synthetic/anonymized dataset was used for demonstration.